---
downloads:
  - file: session1_data_acquisition.ipynb
    title: Jupyter Notebook (.ipynb)
  - file: ../downloads/books.csv
    title: Scraped Books-Data (.csv)
  - id: lecture-script
    title: Lecture Script (.pdf)
---

# Session 1 - Data Acquisition and Web Scraping

Date: 09.04.2026

# Why this session matters

In Data Science, a model is only as good as the data pipeline behind it.

Today we focus on how to *get data well* before we model anything.

We will work with ideas inspired by the legacy lecture material:
- workflow thinking
- data quality awareness
- practical scraping and API access

# Learning goals

By the end of this notebook you can:

- collect tabular data from web pages with `pandas.read_html`
- parse HTML documents with `BeautifulSoup`
- extract useful fields such as links, titles, and structured snippets
- compare scraping approaches (tables vs HTML parsing vs APIs)
- set up a simple human-in-the-loop labeling pipeline with Label Studio
- apply a small quality checklist before using acquired data

# Setup

If needed, install once:

In [ ]:
pip install pandas beautifulsoup4 requests scikit-learn lxml selenium

# Datasets

## Public Data APIs

Often data for a given task can be found in public data sources, such as:

- https://daten.berlin.de/
- https://www.kaggle.com/datasets
- https://datasetsearch.research.google.com/

Direct downloads are often the fastest first step before scraping.

## Public ML Data Sets: OpenML

A practical source for benchmark datasets with a Python API. Joaquin Vanschoren set up https://www.openml.org/
- over 20,000 data sets
- useful for model prototyping
- metadata is usually better than ad-hoc scraping

In [6]:
from sklearn.datasets import fetch_openml

X, y = fetch_openml("titanic", as_frame=True, return_X_y=True, version=1)
print(f"Rows: {X.shape[0]}, Cols: {X.shape[1]}")
X.head()

Rows: 1309, Cols: 13


,pclass,name,sex,age,sibsp,parch,ticket,fare,cabin,embarked,boat,body,home.dest
0,1,"Allen, Miss. Elisabeth Walton",female,29.0000,0,0,24160,211.3375,B5,S,2,NaN,"St Louis, MO"
1,1,"Allison, Master. Hudson Trevor",male,0.9167,1,2,113781,151.5500,C22 C26,S,11,NaN,"Montreal, PQ / Chesterville, ON"
2,1,"Allison, Miss. Helen Loraine",female,2.0000,1,2,113781,151.5500,C22 C26,S,NaN,NaN,"Montreal, PQ / Chesterville, ON"
3,1,"Allison, Mr. Hudson Joshua Creighton",male,30.0000,1,2,113781,151.5500,C22 C26,S,NaN,135.0,"Montreal, PQ / Chesterville, ON"
4,1,"Allison, Mrs. Hudson J C (Bessie Waldo Daniels)",female,25.0000,1,2,113781,151.5500,C22 C26,S,NaN,NaN,"Montreal, PQ / Chesterville, ON"


# Part 1: Tabular scraping with pandas

Just like in some previous lectures, we start with Wikipedia tables.

Run the next cell, then choose a URL and inspect available tables interactively.

In [27]:
import pandas as pd
import requests
from io import StringIO
from IPython.display import display

def preview_table_source(url: str, table_idx: int = 0, n_rows: int = 8):
    # Wikipedia started blocking requests without a user-agent header, so we need to set it to mimic a browser
    headers = {
        "User-Agent": "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36",
        "Accept-Language": "en-US,en;q=0.9",
    }
    response = requests.get(url, headers=headers, timeout=20)
    response.raise_for_status()

    # Parse HTML text
    tables = pd.read_html(StringIO(response.text))
    print(f"Found {len(tables)} tables at: {url}")
    idx = max(0, min(table_idx, len(tables) - 1))
    print(f"Showing table {idx} ({tables[idx].shape[0]} rows, {tables[idx].shape[1]} columns):")
    display(tables[idx].head(n_rows))

preview_table_source("https://en.wikipedia.org/wiki/Berlin_population_statistics", table_idx=0, n_rows=5)

Found 7 tables at: https://en.wikipedia.org/wiki/Berlin_population_statistics
Showing table 0 (13 rows, 4 columns):


,Borough,Population 30 September 2010,Area in km2,Largest Non-German ethnic groups
0,Mitte,332100,39.47,"Turks, Arabs, Kurds, many Asians, Africans and..."
1,Friedrichshain-Kreuzberg,268831,20.16,"Turks, Arabs, African, Kurds, Chinese"
2,Pankow,368956,103.01,"Poles, Italians, French, Americans, Vietnamese..."
3,Charlottenburg-Wilmersdorf,320014,64.72,"Turks, Africans, Russians, Arabs, others."
4,Spandau,225420,91.91,"Turks, Africans, Russians, Arabs, others."


# Part 2: HTML parsing with BeautifulSoup

Now we move to more sophisticated data scraping not just from tables, but from the HTML page structure. 
- A great playground to test scraping: https://toscrape.com
- We will use: https://books.toscrape.com to extract book titles, prices, ratings and descriptions.

In [93]:
import requests
from bs4 import BeautifulSoup

url = "https://books.toscrape.com"
response = requests.get(url, timeout=20)
response.raise_for_status()
soup = BeautifulSoup(response.text, "html.parser")

print(f"Title: {soup.title.get_text(strip=True) if soup.title else '(no title)'}")

Title: All products | Books to Scrape - Sandbox


In [ ]:
# Extract single product information
product_pods = soup.select("article.product_pod")
single_product = product_pods[0]
print(single_product.prettify())

```html
<article class="product_pod">
 <div class="image_container">
  <a href="catalogue/a-light-in-the-attic_1000/index.html">
   <img alt="A Light in the Attic" class="thumbnail" src="media/cache/2c/da/2cdad67c44b002e7ead0cc35693c0e8b.jpg"/>
  </a>
 </div>
 <p class="star-rating Three">
  <i class="icon-star">
  </i>
  <i class="icon-star">
  </i>
  <i class="icon-star">
  </i>
  <i class="icon-star">
  </i>
  <i class="icon-star">
  </i>
 </p>
 <h3>
  <a href="catalogue/a-light-in-the-attic_1000/index.html" title="A Light in the Attic">
   A Light in the ...
  </a>
 </h3>
 <div class="product_price">
  <p class="price_color">
   Â£51.77
  </p>
  <p class="instock availability">
   <i class="icon-ok">
   </i>
   In stock
  </p>
  <form>
   <button class="btn btn-primary btn-block" data-loading-text="Adding..." type="submit">
    Add to basket
   </button>
  </form>
 </div>
</article>
```

## Selector Syntax
Basics:
- ``tag`` -> all elements of that tag<br>
Example: ``p``
- ``.class`` -> elements with a class<br>
Example: ``.price_color``
- ``#id`` -> element with a specific id<br>
Example: ``#product_description``

Combinations:
- ``A B`` -> descendant of A (any depth)<br>
Example: ``article p``
- ``A > B`` -> direct child of A<br>
Example: ``article > p``
- ``A + B`` -> immediate next sibling<br>
Example: ``h2 + p``
- ``A ~ B`` -> any following sibling<br>
Example: ``#product_description ~ p``

Attribute Matching:
- ``[attr="value"]`` -> has attribute<br>
Example: ``a[id="product_description"]``

In [102]:
# Extract title information
title_element = single_product.select_one("h3 a")
title_element

<a href="catalogue/a-light-in-the-attic_1000/index.html" title="A Light in the Attic">A Light in the ...</a>

In [103]:
print(f"From tag: {title_element.get("title")}")
print(f"From text: {title_element.get_text()}")

From tag: A Light in the Attic
From text: A Light in the ...


In [110]:
# Get description from subpage
href = title_element.get("href")
response = requests.get(f"{url}/{href}", timeout=20)
response.raise_for_status()

subsoup = BeautifulSoup(response.text, "html.parser")
description = subsoup.select_one("div[id='product_description'] + p").get_text(strip=True)
description

"It's hard to imagine a world without A Light in the Attic. This now-classic collection of poetry and drawings from Shel Silverstein celebrates its 20th anniversary with this special edition. Silverstein's humorous and creative verse can amuse the dowdiest of readers. Lemon-faced adults and fidgety kids sit still and read these rhythmic words and laugh and smile and love th It's hard to imagine a world without A Light in the Attic. This now-classic collection of poetry and drawings from Shel Silverstein celebrates its 20th anniversary with this special edition. Silverstein's humorous and creative verse can amuse the dowdiest of readers. Lemon-faced adults and fidgety kids sit still and read these rhythmic words and laugh and smile and love that Silverstein. Need proof of his genius? RockabyeRockabye baby, in the treetopDon't you know a treetopIs no safe place to rock?And who put you up there,And your cradle, too?Baby, I think someone down here'sGot it in for you. Shel, you never sounde

Lets bring it together now:

In [113]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
from IPython.display import display

url = "https://books.toscrape.com"
response = requests.get(url, timeout=20)
response.raise_for_status()
soup = BeautifulSoup(response.text, "html.parser")

def extract_books_single_page(soup: BeautifulSoup, url: str):
    # Retrieve all books
    product_pods = soup.select("article.product_pod")

    # Dictionary in which we fill our data
    scraped_data = {"Title": [], "Price": [], "Availability": [], "Rating": [], "Description": []}

    # Loop over the products to extract
    for product in product_pods:
        # Info directly on main page
        title = product.select_one("h3 a").get("title")
        price = product.select_one("div.product_price p.price_color").get_text()[1:]
        stock = product.select_one("div.product_price p.instock.availability").get_text().strip()
        rating = product.select_one("p").get("class")[1]

        # Info on subpage
        href = product.select_one("h3 a").get("href")
        response = requests.get(url=f"{url}/{href}", timeout=3)
        response.raise_for_status()
        subsoup = BeautifulSoup(response.text, "html.parser")
        description = subsoup.select_one("article.product_page > p").get_text() # Sometime you might need ">" to enforce direct child selection
        
        scraped_data["Title"].append(title)
        scraped_data["Price"].append(price)
        scraped_data["Availability"].append(stock)
        scraped_data["Rating"].append(rating)
        scraped_data["Description"].append(description)
    return scraped_data

scraped_data = pd.DataFrame(extract_books_single_page(soup, url))

display(scraped_data.head())

Title: All products | Books to Scrape - Sandbox


,Title,Price,Availability,Rating,Description
0,A Light in the Attic,£51.77,In stock,Three,It's hard to imagine a world without A Light i...
1,Tipping the Velvet,£53.74,In stock,One,"""Erotic and absorbing...Written with starling ..."
2,Soumission,£50.10,In stock,One,"Dans une France assez proche de la nÃ´tre, un ..."
3,Sharp Objects,£47.82,In stock,Four,"WICKED above her hipbone, GIRL across her hear..."
4,Sapiens: A Brief History of Humankind,£54.23,In stock,Five,From a renowned historian comes a groundbreaki...


In [114]:
scraped_data.to_csv("books.csv", index=False)

# Part 3: Circumvent anti-scraping measures with Selenium
Some websites use measures, that make it difficult to scrape simply with requests and BeautifulSoup. For example, by:
- dynamic content loading
- pagination
- bot detection

For this, we can use Selenium, to automate a browser that can interact with the page as a human would. Selenium has many features such as:
- waiting for elements to load
- clicking buttons and links
- filling out forms
- handling cookies and sessions

In [ ]:
# TODO: SELENIUM EXAMPLE

# Part 4: Human-in-the-loop annotation (Label Studio)

For many tasks, labels are not available and must be created manually. For this we can use a tool like Label Studio (https://labelstud.io/) to set up a simple annotation pipeline. With Label Studio, you can define labeling schemas for various data types (text, images, tables).

## Install and run Label Studio

```bash
pip install label-studio
label-studio
```
(runs on port 8080 by default: http://localhost:8080)

If you encounter issues with importerror pkgutil, try:

```bash
pip install -U django-environ
```

Then open the shown URL, create a new project, and import tasks from the next cell output.

## Labeling Demonstration

Download the scraped book data from the Download section and import it as tasks in Label Studio. Use the following label configuration:

```xml
<View>
  <Header value="Book Record"/>

  <HyperText name="table" value="&lt;div style='font-family: -apple-system, BlinkMacSystemFont, Segoe UI, Roboto, Helvetica Neue, Arial, sans-serif;'&gt;
    &lt;table style='width:100%; border-collapse:collapse; font-size:14px;'&gt;
      &lt;tr&gt;&lt;th style='text-align:left; padding:8px; border:1px solid #ddd; width:160px;'&gt;Field&lt;/th&gt;&lt;th style='text-align:left; padding:8px; border:1px solid #ddd;'&gt;Value&lt;/th&gt;&lt;/tr&gt;
      &lt;tr&gt;&lt;td style='padding:8px; border:1px solid #ddd;'&gt;&lt;strong&gt;Title&lt;/strong&gt;&lt;/td&gt;&lt;td style='padding:8px; border:1px solid #ddd;'&gt;$Title&lt;/td&gt;&lt;/tr&gt;
      &lt;tr&gt;&lt;td style='padding:8px; border:1px solid #ddd;'&gt;&lt;strong&gt;Price&lt;/strong&gt;&lt;/td&gt;&lt;td style='padding:8px; border:1px solid #ddd;'&gt;$Price&lt;/td&gt;&lt;/tr&gt;
      &lt;tr&gt;&lt;td style='padding:8px; border:1px solid #ddd;'&gt;&lt;strong&gt;Availability&lt;/strong&gt;&lt;/td&gt;&lt;td style='padding:8px; border:1px solid #ddd;'&gt;$Availability&lt;/td&gt;&lt;/tr&gt;
      &lt;tr&gt;&lt;td style='padding:8px; border:1px solid #ddd;'&gt;&lt;strong&gt;Rating&lt;/strong&gt;&lt;/td&gt;&lt;td style='padding:8px; border:1px solid #ddd;'&gt;$Rating&lt;/td&gt;&lt;/tr&gt;
      &lt;tr&gt;&lt;td style='padding:8px; border:1px solid #ddd; vertical-align:top;'&gt;&lt;strong&gt;Description&lt;/strong&gt;&lt;/td&gt;&lt;td style='padding:8px; border:1px solid #ddd;'&gt;$Description&lt;/td&gt;&lt;/tr&gt;
    &lt;/table&gt;
  &lt;/div&gt;"/>

  <Header value="Target Group"/>
  <Choices name="target_group" toName="table" choice="single" showInline="true">
    <Choice value="Kids"/>
    <Choice value="Teens"/>
    <Choice value="Adults"/>
    <Choice value="Unclear"/>
  </Choices>

  <Header value="Compellingness"/>
  <Choices name="compellingness" toName="table" choice="single" showInline="true">
    <Choice value="Compelling"/>
    <Choice value="Not Compelling"/>
  </Choices>
</View>
```

This is how it looks in Label Studio:

![Label Studio annotation interface](../figures/book-records-labelstudio.png)

# Exercise

Choose one website and create a small scraper that returns a clean DataFrame.
TODO: Some more here.